# Pygformula pipeline with features identiying L

In [1]:
import pandas as pd
import numpy as np
import pygformula
from pygformula import ParametricGformula
from pygformula.interventions import static
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb

In [2]:
import re

In [3]:
#%matplotlib ipympl

# Load MIMIC data

In [4]:
mimicdata = pd.read_parquet('../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet')

In [5]:
mimicdata.shape

(599188, 42)

In [6]:
mimicdata.columns

Index(['stay_id', 'grid_end', 'vent_mode__hours_since_last__last_12h',
       'temperature__mean__last_12h', 'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'o2_flow__last__last_12h',
       'vent_mode__last__last_12h', 'subject_id', 'raw_age', 'sex',
       'raw_height', 'raw_weight', 'unit_type', 'origin', 'los', 'intime',
       'outtime', 'death_time_from_intime', 'icu_mortality_flag',
       'death_abs_time', 'mortality_after_discharge', 'icu_mortality', 't',
       

In [7]:
mimicdata.rename(columns={"stay_id": "admission_id", "t": "t0", "raw_age":"age", "raw_height":"height", "raw_weight":"weight"}, inplace=True)

In [8]:
mimicdata.vent_mode__last__last_12h = mimicdata.vent_mode__last__last_12h.astype("category")

In [9]:
mimicdata.vent_mode__last__last_12h.value_counts()

vent_mode__last__last_12h
unknown                522887
invasive_controlled     56730
invasive_assisted       19525
cancelled                  46
Name: count, dtype: int64

In [10]:
mimicdata.o2_flow__last__last_12h.describe()

count    45301.000000
mean        46.678171
std         13.793587
min          7.342648
25%         40.000000
50%         40.000000
75%         50.000000
max         87.342648
Name: o2_flow__last__last_12h, dtype: float64

# Declare baseline variables and meta variables

In [11]:
#baseline = ['age',
#            'gender',
#            'ethnicity',
#            'insurance'
#            ]
baseline = ["age", 
            "sex", 
            #"height", 
            #"weight", 
            #"unit_type", 
            "origin"]

# Mortality

In [12]:
obs_data = mimicdata.sort_values(["admission_id", "t0"]).copy()

# Initialize Y as NaN everywhere
obs_data["Y"] = np.nan

# Identify last row per stay
last_idx = obs_data.groupby(id, sort=False).tail(1).index

# Extract needed columns on last rows
A_last = obs_data.loc[last_idx, "A"]
I_last = obs_data.loc[last_idx, "D"]
Z_last = obs_data.loc[last_idx, "Z"]

# Compute Y_hat according to your rule (without filling NaNs globally)
Y_last = (
    (I_last == 1) |
    ((A_last == 1) & (Z_last == 1))
).astype(int)

# Assign only to terminal rows
obs_data.loc[last_idx, "Y"] = Y_last
obs_mean_Ey = obs_data.loc[last_idx, "Y"].mean()
obs_mean_Ey

np.float64(0.02909270546139108)

In [13]:
mimicdata.groupby(['A', 'D', 'Z', 'Y'], dropna=False).size().reset_index(name='count')

,A,D,Z,Y,count
0,0,0.0,NaN,0.0,13
1,0,0.0,NaN,NaN,513542
2,0,1.0,NaN,1.0,6316
3,1,NaN,0.0,0.0,68201
4,1,NaN,1.0,1.0,11116


In [14]:
picked_L = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'bicarbonate_arterial__last__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'temperature__mean__last_12h',

    'hemoglobin__last__last_12h',
    'heart_rate__mean__last_12h',
    'arterial_blood_pressure_mean__mean__last_12h',

    'creatinine__last__last_12h',
    'ureum__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'lactate__last__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    'o2_saturation__mean__last_12h',
    'o2_flow__last__last_12h',  
    'po2_arterial__mean__last_12h',
]

In [15]:
mimicdata.admission_id.nunique()

85646

## Handle missing values

In [16]:
# Columns to forward fill
feature_columns = list(picked_L)

In [17]:
mimicdata[feature_columns].isna().sum()

vent_mode__last__last_12h                                     0
vent_mode__hours_since_last__last_12h                    453814
bicarbonate_arterial__last__last_12h                     582973
activated_partial_thromboplastin_time__last__last_12h    588525
temperature__mean__last_12h                               48046
hemoglobin__last__last_12h                               582830
heart_rate__mean__last_12h                                 2199
arterial_blood_pressure_mean__mean__last_12h              16526
creatinine__last__last_12h                               582967
ureum__last__last_12h                                    582980
fluid_out_urine__mean__last_12h                          202848
lactate__last__last_12h                                  589939
glasgow_coma_scale_total__last__last_12h                 516053
pco2_arterial__mean__last_12h                            370415
respiratory_rate_measured__mean__last_12h                  5107
o2_saturation__mean__last_12h           

In [18]:
# # Apply forward fill per group, only on selected columns; Then backward fill to handle top-level NaNs per group
mimicdata[feature_columns] = mimicdata.groupby('admission_id')[feature_columns].ffill().bfill()

In [19]:
mimicdata[feature_columns].isna().sum()

vent_mode__last__last_12h                                 0
vent_mode__hours_since_last__last_12h                    25
bicarbonate_arterial__last__last_12h                     47
activated_partial_thromboplastin_time__last__last_12h    47
temperature__mean__last_12h                               0
hemoglobin__last__last_12h                               47
heart_rate__mean__last_12h                                0
arterial_blood_pressure_mean__mean__last_12h              0
creatinine__last__last_12h                               47
ureum__last__last_12h                                    47
fluid_out_urine__mean__last_12h                          16
lactate__last__last_12h                                  87
glasgow_coma_scale_total__last__last_12h                  0
pco2_arterial__mean__last_12h                            25
respiratory_rate_measured__mean__last_12h                 0
o2_saturation__mean__last_12h                             0
o2_flow__last__last_12h                 

In [20]:
# Find rows where any of these columns are NaN
mask_missing = mimicdata[feature_columns].isna().any(axis=1)

# Get corresponding admission IDs
missing_admissions = mimicdata.loc[mask_missing, "admission_id"].unique()

print("Admissions with any missing feature:", missing_admissions)
print("Total:", len(missing_admissions))


Admissions with any missing feature: ['mimic4-39998138' 'mimic4-39998622' 'mimic4-39999168' 'mimic4-39999172'
 'mimic4-39999230' 'mimic4-39999286' 'mimic4-39999301' 'mimic4-39999384'
 'mimic4-39999552' 'mimic4-39999562' 'mimic4-39999810' 'mimic4-39999858']
Total: 12


In [21]:
mimicdata.loc[mimicdata['admission_id']=='mimic4-39941251'].isna().sum()

admission_id                                             0
grid_end                                                 0
vent_mode__hours_since_last__last_12h                    0
temperature__mean__last_12h                              0
heart_rate__mean__last_12h                               0
arterial_blood_pressure_mean__mean__last_12h             0
fluid_out_urine__mean__last_12h                          0
pco2_arterial__mean__last_12h                            0
respiratory_rate_measured__mean__last_12h                0
o2_saturation__mean__last_12h                            0
po2_arterial__mean__last_12h                             0
bicarbonate_arterial__last__last_12h                     0
activated_partial_thromboplastin_time__last__last_12h    0
hemoglobin__last__last_12h                               0
creatinine__last__last_12h                               0
ureum__last__last_12h                                    0
lactate__last__last_12h                                 

In [22]:
mimicdata = mimicdata[~mimicdata['admission_id'].isin(missing_admissions)]

# Check missingness including baseline variables

In [23]:
#check missingness including baseline variables
all = picked_L + baseline
mimicdata[all].isna().sum()

vent_mode__last__last_12h                                    0
vent_mode__hours_since_last__last_12h                        0
bicarbonate_arterial__last__last_12h                         0
activated_partial_thromboplastin_time__last__last_12h        0
temperature__mean__last_12h                                  0
hemoglobin__last__last_12h                                   0
heart_rate__mean__last_12h                                   0
arterial_blood_pressure_mean__mean__last_12h                 0
creatinine__last__last_12h                                   0
ureum__last__last_12h                                        0
fluid_out_urine__mean__last_12h                              0
lactate__last__last_12h                                      0
glasgow_coma_scale_total__last__last_12h                     0
pco2_arterial__mean__last_12h                                0
respiratory_rate_measured__mean__last_12h                    0
o2_saturation__mean__last_12h                          

In [24]:
# Find rows where any of these columns are NaN
mask_missing = mimicdata[all].isna().any(axis=1)

# Get corresponding admission IDs
missing_admissions = mimicdata.loc[mask_missing, "admission_id"].unique()

print("Admissions with any missing feature:", missing_admissions)
print("Total:", len(missing_admissions))

Admissions with any missing feature: ['mimic4-30001555' 'mimic4-30004232' 'mimic4-30004320' ...
 'mimic4-39986935' 'mimic4-39987185' 'mimic4-39997370']
Total: 3154


In [25]:
mimicdata.loc[mimicdata.admission_id=="mimic4-30001555"].sex

35   NaN
36   NaN
Name: sex, dtype: float64

In [26]:
mimicdata = mimicdata[~mimicdata['admission_id'].isin(missing_admissions)]

In [27]:
mimicdata[all].isna().sum()

vent_mode__last__last_12h                                0
vent_mode__hours_since_last__last_12h                    0
bicarbonate_arterial__last__last_12h                     0
activated_partial_thromboplastin_time__last__last_12h    0
temperature__mean__last_12h                              0
hemoglobin__last__last_12h                               0
heart_rate__mean__last_12h                               0
arterial_blood_pressure_mean__mean__last_12h             0
creatinine__last__last_12h                               0
ureum__last__last_12h                                    0
fluid_out_urine__mean__last_12h                          0
lactate__last__last_12h                                  0
glasgow_coma_scale_total__last__last_12h                 0
pco2_arterial__mean__last_12h                            0
respiratory_rate_measured__mean__last_12h                0
o2_saturation__mean__last_12h                            0
o2_flow__last__last_12h                                 

In [28]:
mimicdata['t0'] = mimicdata['t0'].astype(int)

# Normalizing numeric columns

In [29]:
los_per_admission = mimicdata.groupby("admission_id")["t0"].max().reset_index(name="LOS")
los_per_admission["LOS"] = los_per_admission["LOS"] + 1
unique_los = los_per_admission["LOS"].unique()
unique_los.sort()
print(unique_los)

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  95  96  97  98  99 100 101 102 103 104 105 106 107 108 109
 110 111 113 114 115 116 117 118 119 121 123 125 126 127 129 132 134 135
 137 139 140 143 144 147 148 149 150 151 154 155 157 162 163 165 173 175
 181]


In [30]:
# LOS per admission (assuming t0 starts at 0)
los_per_admission = (
    mimicdata.groupby("admission_id")["t0"].max().reset_index(name="LOS")
)
los_per_admission["LOS"] = los_per_admission["LOS"] + 1

# Distribution of LOS
los_distribution = los_per_admission["LOS"].value_counts().sort_index()

print(los_distribution)

LOS
1      18638
2      12314
3      11648
4       6279
5       6299
       ...  
163        1
165        2
173        1
175        1
181       12
Name: count, Length: 145, dtype: int64


In [31]:
mimicdata.admission_id.nunique()

82480

In [32]:
# Cut O2 saturation to 100%
mimicdata["o2_saturation__mean__last_12h"] = (
    mimicdata["o2_saturation__mean__last_12h"].clip(upper=100)
)

# Crafting discharge policies

In [33]:
# variable list
covnames = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'po2_arterial__mean__last_12h',
    'o2_flow__last__last_12h',
    'o2_saturation__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',

    'lactate__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'ureum__last__last_12h',
    'creatinine__last__last_12h',

    'arterial_blood_pressure_mean__mean__last_12h',
    'heart_rate__mean__last_12h',
    'hemoglobin__last__last_12h',
    
    'temperature__mean__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'bicarbonate_arterial__last__last_12h',
    
    'A'
]

cat = ['vent_mode__last__last_12h']

def summarize_variable(series, ndigits=2, is_categorical=False):
    
    # Categorical variable
    if is_categorical:
        value_counts = series.value_counts(dropna=False)
        return {
            "counts": value_counts.to_dict(),
        }
    
    # Numeric variable
    if not series.notna().any():
        return {"interval": None, "mean": None, "median": None}

    min_v = float(series.min(skipna=True))
    max_v = float(series.max(skipna=True))
    mean_v = float(series.mean(skipna=True))
    med_v = float(series.median(skipna=True))

    return {
        "interval": f"{round(min_v, ndigits)} – {round(max_v, ndigits)}",
        "mean": round(mean_v, ndigits),
        "median": round(med_v, ndigits)
    }


# Filter to discharge events
df_discharge = mimicdata[mimicdata["A"] == 1]

vars_to_summarize = [v for v in covnames if v != "A"]

ranges_dict = {
    var: summarize_variable(
            df_discharge[var],
            ndigits=2,
            is_categorical=(var in cat)
         )
    for var in vars_to_summarize
}

ranges_dict

{'vent_mode__last__last_12h': {'counts': {'unknown': 74171,
   'invasive_assisted': 1340,
   'invasive_controlled': 867,
   'cancelled': 2}},
 'vent_mode__hours_since_last__last_12h': {'interval': '0.0 – 12.0',
  'mean': 4.2,
  'median': 3.27},
 'pco2_arterial__mean__last_12h': {'interval': '0.0 – 91.12',
  'mean': 42.12,
  'median': 41.0},
 'po2_arterial__mean__last_12h': {'interval': '12.33 – 336.02',
  'mean': 120.75,
  'median': 98.5},
 'o2_flow__last__last_12h': {'interval': '7.34 – 87.34',
  'mean': 48.13,
  'median': 40.0},
 'o2_saturation__mean__last_12h': {'interval': '83.98 – 100.0',
  'mean': 96.35,
  'median': 96.46},
 'respiratory_rate_measured__mean__last_12h': {'interval': '0.0 – 43.92',
  'mean': 19.0,
  'median': 18.5},
 'glasgow_coma_scale_total__last__last_12h': {'interval': '3.0 – 15.0',
  'mean': 12.62,
  'median': 15.0},
 'lactate__last__last_12h': {'interval': '0.3 – 8.12',
  'mean': 2.29,
  'median': 1.6},
 'fluid_out_urine__mean__last_12h': {'interval': '0.0 – 

In [34]:
mimicdata.admission_id.nunique()

82480

In [35]:
# Sub sample of admissions
rng = np.random.default_rng(34)
unique_ids = mimicdata['admission_id'].dropna().unique()
sampled_ids = rng.choice(unique_ids, size=5000, replace=False)
mimicdata_sub = mimicdata[mimicdata['admission_id'].isin(sampled_ids)].copy()
mimicdata_sub.reset_index(drop=True, inplace=True)

In [36]:
#mimicdata.to_parquet("gdata.parquet")

In [37]:
mimicdata.admission_id.nunique()

82480

# Applying g-formula

In [38]:
time_name = 't0'
id_ = 'admission_id'

In [39]:
basecovs = baseline
'''['age',
            'gender',
            'ethnicity',
            'insurance'
            ]'''

"['age',\n            'gender',\n            'ethnicity',\n            'insurance'\n            ]"

In [40]:
covnames = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'po2_arterial__mean__last_12h',
    'o2_flow__last__last_12h',
    'o2_saturation__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',

    'lactate__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'ureum__last__last_12h',
    'creatinine__last__last_12h',

    'arterial_blood_pressure_mean__mean__last_12h',
    'heart_rate__mean__last_12h',
    'hemoglobin__last__last_12h',
    
    'temperature__mean__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'bicarbonate_arterial__last__last_12h',
    
    'A'
]

In [41]:
covtypes = [
 'custom',          #'vent_mode__last__last_12h'
 'custom', #'zero-inflated normal', #'vent_mode__hours_since_last__last_12h'

 'custom',       #'pco2_arterial__mean__last_12h'
 'custom',       #'po2_arterial__mean__last_12h'
 'custom',       #'o2_flow__last__last_12h',
 'custom',       #'o2_saturation__mean__last_12h',
 'custom',       #'respiratory_rate_measured__mean__last_12h',

 'custom',       #'glasgow_coma_scale_total__last__last_12h',
 'custom',     #'lactate__last__last_12h',

 'custom', #'fluid_out_urine__mean__last_12h',
 'custom',     #'ureum__last__last_12h',
 'custom',     #'creatinine__last__last_12h',

 'custom',       #'arterial_blood_pressure_mean__mean__last_12h',
 'custom',       #'heart_rate__mean__last_12h',
 'custom',       #'hemoglobin__last__last_12h'
    
 'custom',               #'temperature__mean__last_12h',
 'custom',     #'activated_partial_thromboplastin_time__last__last_12h',
 'custom',       #'bicarbonate_arterial__last__last_12h'

 'custom'              #'A'
]

In [42]:
covmodels = [ 
 'vent_mode__last__last_12h ~ glasgow_coma_scale_total__last__last_12h + po2_arterial__mean__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h',    
 
 'vent_mode__hours_since_last__last_12h ~ lag1_vent_mode__hours_since_last__last_12h + vent_mode__last__last_12h',
 #'vent_mode__hours_since_last__last_12h ~ lag1_vent_mode__hours_since_last__last_12h',

 'pco2_arterial__mean__last_12h ~ lag1_pco2_arterial__mean__last_12h + o2_flow__last__last_12h', 
 'po2_arterial__mean__last_12h ~ lag1_po2_arterial__mean__last_12h + o2_flow__last__last_12h + pco2_arterial__mean__last_12h',
 
 'o2_flow__last__last_12h ~ lag1_o2_flow__last__last_12h + vent_mode__last__last_12h + respiratory_rate_measured__mean__last_12h',
 #'o2_flow__last__last_12h ~ lag1_o2_flow__last__last_12h + respiratory_rate_measured__mean__last_12h',
 
 'o2_saturation__mean__last_12h ~ lag1_o2_saturation__mean__last_12h + po2_arterial__mean__last_12h + respiratory_rate_measured__mean__last_12h',
 
 'respiratory_rate_measured__mean__last_12h ~ lag1_respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h',
 #'respiratory_rate_measured__mean__last_12h ~ lag1_respiratory_rate_measured__mean__last_12h',

 'glasgow_coma_scale_total__last__last_12h ~ lag1_glasgow_coma_scale_total__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + lactate__last__last_12h',
 
 #'lactate__last__last_12h ~ lag1_lactate__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + fluid_out_urine__mean__last_12h',
 'lactate__last__last_12h ~ lag1_lactate__last__last_12h + arterial_blood_pressure_mean__mean__last_12h',

 'fluid_out_urine__mean__last_12h ~ lag1_fluid_out_urine__mean__last_12h + arterial_blood_pressure_mean__mean__last_12h + ureum__last__last_12h',
 'ureum__last__last_12h ~ lag1_ureum__last__last_12h + creatinine__last__last_12h',
 'creatinine__last__last_12h ~ lag1_creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h',

 'arterial_blood_pressure_mean__mean__last_12h ~ lag1_arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + bicarbonate_arterial__last__last_12h',
 'heart_rate__mean__last_12h ~ lag1_heart_rate__mean__last_12h + temperature__mean__last_12h + hemoglobin__last__last_12h',
 'hemoglobin__last__last_12h ~ lag1_hemoglobin__last__last_12h + activated_partial_thromboplastin_time__last__last_12h',
    
 'temperature__mean__last_12h ~ lag1_temperature__mean__last_12h',
 'activated_partial_thromboplastin_time__last__last_12h ~ lag1_activated_partial_thromboplastin_time__last__last_12h',
 'bicarbonate_arterial__last__last_12h ~ lag1_bicarbonate_arterial__last__last_12h',

 
 'A ~ vent_mode__last__last_12h + vent_mode__hours_since_last__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h']
 #'A ~ vent_mode__hours_since_last__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h']


In [43]:
outcome_name = 'Z'
# Outcome model
zmodel = 'Z ~ vent_mode__last__last_12h + cumavg_vent_mode__hours_since_last__last_12h + cumavg_pco2_arterial__mean__last_12h + cumavg_po2_arterial__mean__last_12h + cumavg_o2_flow__last__last_12h + cumavg_o2_saturation__mean__last_12h + cumavg_respiratory_rate_measured__mean__last_12h + cumavg_glasgow_coma_scale_total__last__last_12h + cumavg_lactate__last__last_12h + cumavg_fluid_out_urine__mean__last_12h + cumavg_ureum__last__last_12h + cumavg_creatinine__last__last_12h + cumavg_arterial_blood_pressure_mean__mean__last_12h + cumavg_heart_rate__mean__last_12h + cumavg_hemoglobin__last__last_12h + cumavg_temperature__mean__last_12h + cumavg_activated_partial_thromboplastin_time__last__last_12h + cumavg_bicarbonate_arterial__last__last_12h'
#zmodel = 'Z ~ cumavg_vent_mode__hours_since_last__last_12h + cumavg_pco2_arterial__mean__last_12h + cumavg_po2_arterial__mean__last_12h + cumavg_o2_flow__last__last_12h + cumavg_o2_saturation__mean__last_12h + cumavg_respiratory_rate_measured__mean__last_12h + cumavg_glasgow_coma_scale_total__last__last_12h + cumavg_lactate__last__last_12h + cumavg_fluid_out_urine__mean__last_12h + cumavg_ureum__last__last_12h + cumavg_creatinine__last__last_12h + cumavg_arterial_blood_pressure_mean__mean__last_12h + cumavg_heart_rate__mean__last_12h + cumavg_hemoglobin__last__last_12h + cumavg_temperature__mean__last_12h + cumavg_activated_partial_thromboplastin_time__last__last_12h + cumavg_bicarbonate_arterial__last__last_12h'
outcome_type = 'binary_eof'

In [44]:
I_name = 'D'
I_model = 'D ~ vent_mode__last__last_12h + vent_mode__hours_since_last__last_12h + pco2_arterial__mean__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h'
#I_model = 'D ~ vent_mode__hours_since_last__last_12h + pco2_arterial__mean__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h'

In [45]:
def zmodel_fit_custom(zmodel, fit_data):
    z_name, rhs = re.split(r'~', zmodel.replace(' ', ''))
    x_name = re.split(r'\+', rhs)

    z = fit_data[z_name].to_numpy()
    X = fit_data[x_name].copy()

    # Ensure categorical dtype consistency
    if 'vent_mode__last__last_12h' in X.columns:
        X['vent_mode__last__last_12h'] = X['vent_mode__last__last_12h'].astype('category')

    weights = fit_data["weight"].to_numpy()  # <-- GLM freq_weights equivalent

    z_model_lgbm = lgb.LGBMClassifier(
        objective="binary",
        deterministic=True,
        force_col_wise=True,
        n_jobs=1,
        seed=133,
        feature_fraction_seed=133,
        bagging_seed=133,
        drop_seed=133,
        data_random_seed=133,
    )

    z_model_lgbm.fit(
        X,
        z,
        sample_weight=weights,
        categorical_feature=['vent_mode__last__last_12h']
    )

    return z_model_lgbm

def zmodel_predict_custom(zmodel, new_df, fit):
    z_name, rhs = re.split(r'~', zmodel.replace(' ', ''))
    x_name = re.split(r'\+', rhs)

    # Empty guard (LightGBM errors on empty 2D input)
    if new_df.shape[0] == 0:
        return pd.Series([], dtype=float, index=new_df.index, name=z_name)

    # Use model’s training feature list if available (most robust)
    feat_cols = list(fit.feature_name_) if hasattr(fit, "feature_name_") else x_name

    X = new_df.reindex(columns=feat_cols).copy()

    # Ensure categorical dtype consistency
    if 'vent_mode__last__last_12h' in X.columns:
        X['vent_mode__last__last_12h'] = X['vent_mode__last__last_12h'].astype('category')

    # Predict probabilities
    p = fit.predict_proba(X)[:, 1]

    # Return aligned Series
    return pd.Series(p, index=new_df.index, name=z_name)

## L model for vent_mode__last__last_12h

In [46]:
def fit_ventmode(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMClassifier(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_ventmode(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for vent_mode__hours_since_last__last_12h

In [47]:
def fit_ventmodehours(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=['vent_mode__last__last_12h']  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_ventmodehours(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for pco2

In [48]:
def fit_pco2(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_pco2(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for po2

In [49]:
def fit_po2(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_po2(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for o2flow

In [50]:
def fit_o2flow(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=['vent_mode__last__last_12h']  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_o2flow(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for o2sat

In [51]:
def fit_o2sat(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_o2sat(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model fro rr

In [52]:
def fit_rr(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_rr(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for gcs

In [53]:
def fit_gcs(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_gcs(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for lactate

In [54]:
def fit_lactate(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_lactate(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for fluid

In [55]:
def fit_fluid(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_fluid(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for ureum

In [56]:
def fit_ureum(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_ureum(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for creatinine

In [57]:
def fit_creatinine(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_creatinine(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for abp

In [58]:
def fit_abp(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_abp(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for hr

In [59]:
def fit_hr(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_hr(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for hemoglobin

In [60]:
def fit_hemoglobin(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_hemoglobin(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for temp

In [61]:
def fit_temp(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_temp(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for apTT

In [62]:
def fit_apTT(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_apTT(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# L model for bicarb

In [63]:
def fit_bicarb(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMRegressor(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_bicarb(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

# Model for discharge A

In [64]:
def fit_A(covmodel, covname, fit_data):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    y = fit_data[y_name].to_numpy()
    X = fit_data[x_name]
    
    model_lgbm = lgb.LGBMClassifier(random_state=33)

    # Train the model on data
    model_lgbm.fit(
        X,
        y,
        categorical_feature=[]  # Declare categorical features
        #eval_metric='auc'
    )
    return model_lgbm

def predict_A(covmodel, new_df, fit):
    y_name, x_name = re.split('~', covmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict(X)
    prediction = pd.Series(prediction)
    return prediction

In [65]:
covfits_custom = [
 fit_ventmode,
 fit_ventmodehours,
 fit_pco2,
 fit_po2,
 fit_o2flow,
 fit_o2sat,
 fit_rr,
 fit_gcs,
 fit_lactate,
 fit_fluid,
 fit_ureum,
 fit_creatinine,
 fit_abp,
 fit_hr,
 fit_hemoglobin,
 fit_temp,
 fit_apTT,
 fit_bicarb,
 fit_A
]

covpredict_custom = [
 predict_ventmode,
 predict_ventmodehours,
 predict_pco2,
 predict_po2,
 predict_o2flow,
 predict_o2sat,
 predict_rr,
 predict_gcs,
 predict_lactate,
 predict_fluid,
 predict_ureum,
 predict_creatinine,
 predict_abp,
 predict_hr,
 predict_hemoglobin,
 predict_temp,
 predict_apTT,
 predict_bicarb,
 predict_A
]

In [66]:
time_points = np.max(np.unique(mimicdata[time_name])) + 1
#time_points = 200

In [67]:
time_points

np.int64(181)

In [68]:
#'vent_mode__hours_since_last__last_12h',    
#    'pco2_arterial__mean__last_12h',
#    'po2_arterial__mean__last_12h',
#    'lactate__last__last_12h',
#    'fluid_out_urine__mean__last_12h',
#    'activated_partial_thromboplastin_time__last__last_12h'

# Knight strategy
def dynamic_knight(new_df, pool, int_var, time_name, t):
    new_df.loc[new_df[time_name] == t, int_var] = 0
    condition_knight = (

        # Airway patent
        new_df['vent_mode__last__last_12h'].isin(['unknown']) &

            # Vent status not just "recently changed" (proxy for stability)
            #(new_df['vent_mode__hours_since_last__last_12h'] >= 4.0) &
        
        # --- Respiratory ---
        # R1 FiO2 <= 0.6 
        # (new_df['fio2__mean__last_12h'] <= 0.60) &
        # Proxy: low O2 flow requirement
        (new_df['o2_flow__last__last_12h'] <= 30.0) &

        # R2 SpO2 >= 95 (%)
        (new_df['o2_saturation__mean__last_12h'] >= 95.0) &

        # R3 HCO3 >= 19 (mmol/L)  (your bicarbonate is arterial, mmol/L)
        (new_df['bicarbonate_arterial__last__last_12h'] >= 19.0) &

        # R4 RR 10–30 (bpm)
        (new_df['respiratory_rate_measured__mean__last_12h'].between(10.0, 30.0)) &

        # --- Cardiovascular ---
        # C0 Systolic BP >= 100 mmHg
        # Proxy: MAP >= 75 mmHg
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] >= 75.0) &

        # C1 HR 60–100 bpm
        (new_df['heart_rate__mean__last_12h'].between(60.0, 100.0)) &

        # CNS GCS >= 14
        (new_df['glasgow_coma_scale_total__last__last_12h'] >= 14.0) &

        # T 36–37.5 °C
        (new_df['temperature__mean__last_12h'].between(36.0, 37.5)) &

        # B0 Hb >= 90 g/L
        # Your hemoglobin appears to be in g/dL (mean ~9.9). Convert g/L -> g/dL: 90 g/L = 9.0 g/dL
        (new_df['hemoglobin__last__last_12h'] >= 9.0) &

        # B3 creatinine 59–104 µmol/L
        # Your creatinine appears to be in µmol/L (median ~88). Use as-is.
        (new_df['creatinine__last__last_12h'].between(59.0, 104.0)) &

        # B4 urea (BUN) 2.5–7.8 mmol/L
        # Your variable is "ureum" and appears to be in mmol/L (median ~7.85). Use as-is.
        (new_df['ureum__last__last_12h'].between(2.5, 7.8))
    )

    new_df.loc[condition_knight, int_var] = 1


In [69]:
def dynamic_discharge_aggressive_v3(new_df, pool, int_var, time_name, t):
    # Default: discharge
    new_df.loc[new_df[time_name] == t, int_var] = 1

    # --- ICU-only RED FLAGS (EARLY PHASE) ---
    # Keep this list *minimal*; anything not here is assumed manageable outside ICU (HDU/ward).
    red_flags_early = (
        # 1) Clearly invasive controlled ventilation only
        (new_df['vent_mode__last__last_12h'] == 'invasive_controlled') |

        # 2) Severe oxygenation / support (extremes only)
        (new_df['o2_saturation__mean__last_12h'] < 85.0) |
        (new_df['o2_flow__last__last_12h'] > 85.0) |

        # 3) Profound shock / severe instability (extremes only)
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] < 50.0) |
        (new_df['lactate__last__last_12h'] > 6.0) |
        (new_df['heart_rate__mean__last_12h'] > 160.0) |

        # 4) Deep coma only
        (new_df['glasgow_coma_scale_total__last__last_12h'] < 6.0) |

        # 5) Catastrophic acid–base only
        (new_df['bicarbonate_arterial__last__last_12h'] < 10.0)
    )

    # IMPORTANT: do NOT block just because vent_mode == 'unknown'
    # Only block unknown if physiology suggests ICU-level respiratory failure.
    unknown_vent_extreme = (
        (new_df['vent_mode__last__last_12h'] == 'unknown') &
        (
            (new_df['o2_saturation__mean__last_12h'] < 85.0) |
            (new_df['o2_flow__last__last_12h'] > 85.0) |
            (new_df['pco2_arterial__mean__last_12h'] > 80.0) |
            (new_df['respiratory_rate_measured__mean__last_12h'] > 45.0)
        )
    )

    red_flags_early = red_flags_early | unknown_vent_extreme

    # --- HARD STEP-DOWN DEADLINE (LATE PHASE) ---
    # After this many bins, *force discharge* unless only the MOST extreme life-support needs.
    # Tune: lowering this (e.g., 20–30 bins) usually drops mortality a lot.
    late = (new_df[time_name] == t) & (t >= 24)  # 24 bins = 12 days if bin=12h

    red_flags_late = (
        (new_df['vent_mode__last__last_12h'] == 'invasive_controlled') |
        (new_df['o2_saturation__mean__last_12h'] < 80.0) |
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] < 45.0) |
        (new_df['lactate__last__last_12h'] > 8.0) |
        (new_df['glasgow_coma_scale_total__last__last_12h'] < 5.0)
    )

    keep_in_icu = (red_flags_early & ~late) | (red_flags_late & late)

    new_df.loc[keep_in_icu, int_var] = 0


In [70]:
#int_descript = ['Discharge 0th point', 'Discharge 1st point', 'Discharge 2nd point', 'Discharge 3rd point', 'Discharge 4th point', 'Discharge 5th point', 'Discharge 6th point']

#int_descript = ['Discharge 6th point', 'Discharge natural course', 'Discharge safe', 'Discharge risky', 'Discharge dangerous']
#int_descript = ['Discharge extremely dangerous']
#int_descript = ['Discharge 1st day', 'Discharge 3rd day', 'Discharge 5th day', 'Discharge 7th day', 'Discharge Knight', 'DS1']
#int_descript = ['Discharge 3rd day', 'Discharge safe', 'Discharge risky', 'Discharge dangerous']
#int_descript = ['Discharge 5th day', 'Discharge Knight']

#int_descript = ['Day 1', 'Day 3', 'Day 5', 'Day 7', 'Day 14', 'Day 21', 'Day 28', 'Day 60', 'Day 85']
#int_descript = ['Day 1', 'Day 3', 'Day 5']
int_descript = ['Day 1']

#Intervention1_A = [static, np.array([1]), [0]]
#Intervention1_A = [static, np.array([0, 1]), [0, 1]]
#Intervention3_A = [static, np.array([0, 0, 1]), [0, 1, 2]]
#Intervention4_A = [static, np.array([0, 0, 0, 1]), [0, 1, 2, 3]]
#Intervention5_A = [static, np.array([0, 0, 0, 0, 1]), [0, 1, 2, 3, 4]]
#Intervention6_A = [static, np.array([0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5]]
#Intervention7_A = [static, np.array([0, 0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5, 6]]'''

#Intervention1_A = [static, np.array([0, 0, 0, 0, 0, 0,0,0,0,0,1]), [0, 1, 2, 3, 4, 5, 6,7, 8, 9,10]]

Intervention1_A = [static, np.array([0, 1]), [0, 1]]
#Intervention2_A = [static, np.array([0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5]]
#Intervention3_A = [static, np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]
#Intervention4_A = [static, np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]]

'''# 7 days
Intervention4_A = [
    static,
    np.array([0]*13 + [1]),
    list(range(14))
]

# 14 days
Intervention5_A = [
    static,
    np.array([0]*28 + [1]),
    list(range(29))
]

Intervention6_A = [
    static,
    np.array([0]*42 + [1]),
    list(range(43))
]

Intervention7_A = [
    static,
    np.array([0]*56 + [1]),
    list(range(57))
]

Intervention8_A = [
    static,
    np.array([0]*120 + [1]),
    list(range(121))
]

Intervention9_A = [
    static,
    np.array([0]*170 + [1]),
    list(range(171))
]'''

#int_descript = ['Dynamic intervention']

g = ParametricGformula(obs_data=mimicdata_sub, id=id_, time_name=time_name,
                       covnames=covnames, covtypes=covtypes,
                       covmodels=covmodels, basecovs=basecovs,
                       
                       #trunc_params=trunc_params,
                       
                       int_descript=int_descript,
                       time_points=time_points,
                       
                       Intervention1_A=Intervention1_A,
                       #Intervention2_A=Intervention2_A,
                       #Intervention3_A=Intervention3_A,
                       #Intervention4_A=Intervention4_A,
                       #Intervention5_A=Intervention5_A,
                       #Intervention6_A=Intervention6_A,
                       #Intervention7_A=Intervention7_A,
                       #Intervention8_A=Intervention8_A,
                       #Intervention9_A=Intervention9_A,
                       
                       #Intervention5_A = [dynamic_knight],
                       #Intervention6_A = [dynamic_discharge_aggressive_v3],
                       
                       #Intervention3_A = [dynamic_risky],
                       #Intervention4_A = [dynamic_dangerous],

                       #Intervention1_A=[super_dynamic_dangerous],
                       
                       outcome_name=outcome_name, 
                       zmodel=zmodel,
                       #ymodel=zmodel,
                       outcome_type=outcome_type,
                       
                       zmodel_fit_custom=zmodel_fit_custom, 
                       zmodel_predict_custom=zmodel_predict_custom,

                       covfits_custom = covfits_custom, covpredict_custom=covpredict_custom,

                       
                       I_name=I_name,
                       I_model=I_model,
                       
                       ref_int=0,
                       model_fits=True,
                       n_simul = 500,
                       #censor_name = censor_name, censor_model = censor_model,
                       
                       #nsamples=10, 
                       #parallel=True, ncores=58,
                       #save_path = "results/",
                       #save_results=True,
                       
                       seed = 42,
                       
                       #ipw_cutoff_quantile = 0.99,
                       )

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
{'Day 1': [['A', <function static at 0x7fb967832f20>, array([0, 1]), [0, 1]]], 'Natural course': <function natural at 0x7fb967832e80>}
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


In [71]:
%%time
g.fit()

start fitting parametric model.
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ vent_mode__last__last_12h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003279 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 779
[LightGBM] [Info] Number of data points in the train set: 29409, number of used features: 4
[LightGBM] [Info] Start training from score -8.902762
[LightGBM] [Info] Start training from score -3.310842
[LightGBM] [Info] Start training from score -2.315212
[LightGBM] [Info] Start training from score -0.145450
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ vent_mode__hours_since_last__last_12h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001703 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 29409, number of used features: 2
[LightGBM] [Info] Star

/home/ssimha/Softwares/pygformula/pygformula/parametric_gformula/fit.py:866: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([out1, out2a, out2b, out3], ignore_index=True)


[LightGBM] [Info] Number of positive: 581, number of negative: 4597
[LightGBM] [Info] Total Bins 3428
[LightGBM] [Info] Number of data points in the train set: 5178, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000784 -> initscore=-7.150137
[LightGBM] [Info] Start training from score -7.150137
start simulating.
Time point 0
Time point 1
Time point 2
Time point 3
Time point 4
Time point 5
Time point 6
Time point 7
Time point 8
Time point 9
Time point 10
Time point 11
Time point 12
Time point 13
Time point 14
Time point 15
Time point 16
Time point 17
Time point 18
Time point 19
Time point 20
Time point 21
Time point 22
Time point 23
Time point 24
Time point 25
Time point 26
Time point 27
Time point 28
Time point 29
Time point 30
Time point 31
Time point 32
Time point 33
Time point 34
Time point 35
Time point 36
Time point 37
Time point 38
Time point 39
Time point 40
Time point 41
Time point 42
Time point 43
Time point 44
Time point 45
Time point 46
Time p

/home/ssimha/Softwares/pygformula/pygformula/parametric_gformula/simulate.py:809: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pool = pd.concat(final_df_list, ignore_index=True)


$$$$$$$$$$$$$$$$$$$$$$$##########
[0.796 0.16 ]
$$$$$$$$$$$$$$$$$$$$$$$##########
+---------------------+---------+----------------------------------+----------------+---------------------+
|     Intervention    | NP mean | g-formula mean (NICE-parametric) | Mean Ratio(MR) | Mean Difference(MD) |
+---------------------+---------+----------------------------------+----------------+---------------------+
| Natural course(ref) | 0.19640 |             0.79600              |    1.00000     |       0.00000       |
|        Day 1        |    NA   |             0.16000              |    0.20101     |       -0.63600      |
+---------------------+---------+----------------------------------+----------------+---------------------+
CPU times: user 1h 9min 41s, sys: 11.3 s, total: 1h 9min 52s
Wall time: 2min 39s


# Inspecting time of discharge 

In [84]:
NC = g.summary_dict['sim_data']['Natural course']

In [85]:
NC.columns

Index(['admission_id', 't0', 'vent_mode__last__last_12h',
       'vent_mode__hours_since_last__last_12h',
       'pco2_arterial__mean__last_12h', 'po2_arterial__mean__last_12h',
       'o2_flow__last__last_12h', 'o2_saturation__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'lactate__last__last_12h',
       'fluid_out_urine__mean__last_12h', 'ureum__last__last_12h',
       'creatinine__last__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'heart_rate__mean__last_12h', 'hemoglobin__last__last_12h',
       'temperature__mean__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'bicarbonate_arterial__last__last_12h', 'A', 'age', 'sex', 'origin',
       'lag1_vent_mode__hours_since_last__last_12h',
       'cumavg_vent_mode__hours_since_last__last_12h',
       'lag1_pco2_arterial__mean__last_12h',
       'cumavg_pco2_arterial__mean__last_12h',
       'lag1_po2_arte

# Inspecting Y across LOS

In [75]:
# original data
disc_alive = (mimicdata.groupby("admission_id").tail(1).query("A==1 and D==0")["admission_id"])
Z_orig = mimicdata.groupby("admission_id").tail(1).set_index("admission_id").loc[disc_alive, "Z"].mean()

# simulated NC
disc_alive_sim = NC.loc[NC.A==1]["admission_id"]
Z_sim = NC.loc[NC.admission_id.isin(disc_alive_sim)]["Y_hat"].mean()

print(Z_orig, Z_sim)


nan 0.07339449541284404


In [76]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Y']
             .mean()
)

np.float64(0.20081231813773037)

In [77]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

np.float64(0.9981967213114754)

In [78]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Z']
             .mean()
)

np.float64(0.13713013877978528)

In [79]:
# get last row per admission (with Y and t0)
last_rows = mimicdata.groupby("admission_id").tail(1)[["admission_id", "t0", "Y"]]

# compute LOS (t0 starts at 0, so add 1)
last_rows["LOS"] = last_rows["t0"] + 1

# mean Y per LOS
mean_y_per_los = last_rows.groupby("LOS")["Y"].mean()

print(mean_y_per_los)


LOS
1      0.136978
2      0.142439
3      0.171789
4      0.188884
5      0.196698
         ...   
163    1.000000
165    0.000000
173    0.000000
175    0.000000
181    0.000000
Name: Y, Length: 145, dtype: float64


In [80]:
NC = g.summary_dict['sim_data']['Natural course']

# get last row per admission (with Y and t0)
last_rows = NC.groupby("admission_id").tail(1)[["admission_id", "t0", "Y_hat"]]

# compute LOS (t0 starts at 0, so add 1)
last_rows["LOS"] = last_rows["t0"] + 1

# mean Y per LOS
mean_y_per_los = last_rows.groupby("LOS")["Y_hat"].mean()

print(mean_y_per_los)

LOS
1      0.09009
2      1.00000
3      1.00000
4      1.00000
181    0.00000
Name: Y_hat, dtype: float64


In [81]:
NC = g.summary_dict['sim_data']['Natural course']

In [82]:
NC.groupby('admission_id')['t0'].max().max()

np.int64(180)

In [83]:
DS = g.summary_dict['sim_data']['Discharge Knight']

KeyError: 'Discharge Knight'

In [ ]:
# Compute LOS per admission
los_per_admission = NC.groupby('admission_id')['t0'].max() + 1  # +1 since t0 starts at 0

# Plot histogram
plt.figure(figsize=(8, 5))
los_per_admission.hist(bins=range(1, los_per_admission.max() + 2), edgecolor='black')
plt.xlabel("Length of Stay (number of time points)")
plt.ylabel("Number of Admissions")
plt.title("Distribution of Length of Stay")
plt.grid(True)
plt.tight_layout()
plt.savefig("los_NC.pdf")
plt.show()

In [ ]:
los_per_admission = DS.groupby('admission_id')['t0'].max() + 1

In [ ]:
# Compute LOS (number of time points) per admission
los_per_admission = DS.groupby('admission_id')['t0'].max() + 1  # +1 since t0 starts at 0

# Plot histogram
plt.figure(figsize=(8, 5))
los_per_admission.hist(bins=range(1, los_per_admission.max() + 2), edgecolor='black')
plt.xlabel("Length of Stay (number of time points)")
plt.ylabel("Number of Admissions")
plt.title("Distribution of Length of Stay")
plt.grid(True)
plt.tight_layout()
plt.savefig("los_mimic.pdf")
plt.show()

In [ ]:
DS = g.summary_dict['sim_data']['Discharge safe']
#DR = g.summary_dict['sim_data']['Discharge risky']
#DD = g.summary_dict['sim_data']['Discharge dangerous']

In [ ]:
DS

In [ ]:
# Compute LOS
los_ds = DS.groupby('admission_id')['t0'].max() + 1
#los_dr = DR.groupby('admission_id')['t0'].max() + 1
#los_dd = DD.groupby('admission_id')['t0'].max() + 1

# Create vertically stacked subplots
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 10), sharex=True)

# Plot for DS
axs[0].hist(los_ds, bins=range(1, 200), edgecolor='black', color='skyblue')
axs[0].set_title('LOS Distribution: DS')
axs[0].set_xlabel('Length of Stay (number of time points)')
axs[0].set_ylabel('Number of Admissions')
axs[0].grid(True)

# Plot for DR
axs[1].hist(los_dr, bins=range(1, 200), edgecolor='black', color='salmon')
axs[1].set_title('LOS Distribution: DR')
axs[1].set_xlabel('Length of Stay (number of time points)')
axs[1].set_ylabel('Number of Admissions')
axs[1].grid(True)

# Plot for DD
axs[2].hist(los_dd, bins=range(1, 200), edgecolor='black', color='salmon')
axs[2].set_title('LOS Distribution: DD')
axs[2].set_xlabel('Length of Stay (number of time points)')
axs[2].set_ylabel('Number of Admissions')
axs[2].grid(True)

# Final layout
plt.tight_layout()
plt.savefig("los_comp.pdf")
plt.show()


# Inspect percentage of in-icu death

In [ ]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

In [ ]:
last_rows = (
    NC.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
((last_rows['A'] == 0) & (last_rows['Py'] == 1)).mean()

In [ ]:
last_rows = (
    DS.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
((last_rows['A'] == 0) & (last_rows['Py'] == 1)).mean()

In [ ]:
NC.A

In [ ]:
(
    NC.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

# Mean Y per LOS

In [ ]:
# Step 1: Get the last row for each admission (i.e., end of follow-up)
last_rows = mimicdata.sort_values("t0").groupby("admission_id").tail(1)

# Step 2: Calculate LOS for each admission
last_rows["los"] = mimicdata.groupby("admission_id")["t0"].nunique().values

# Step 3: Group by LOS and compute mean Y
mean_y_per_los = last_rows.groupby("los")["Y"].mean().reset_index()

# Step 4: Display
print(mean_y_per_los)


In [ ]:
mean_y_per_los.Y.mean()

In [ ]:
IPW = g.summary_dict['IP_weights']

In [ ]:
NC

# Distribution of GCS when A=1

In [ ]:
DI = g.summary_dict['sim_data']['Discharge extremely dangerous']

In [ ]:
DI

In [ ]:
# Filter data where A == 1
subset = DI[DI['A'] == 1]

# Plot distribution of X
plt.figure(figsize=(8, 5))
subset['glasgow_coma_scale_total__last__last_12h'].hist(bins=30, edgecolor='black')
plt.xlabel("L values (when A=1)")
plt.ylabel("Frequency")
plt.title("Distribution of X for A=1")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Filter data where A == 1
subset = NC[NC['A'] == 1]

# Plot distribution of X
plt.figure(figsize=(8, 5))
subset['glasgow_coma_scale_total__last__last_12h'].hist(bins=30, edgecolor='black')
plt.xlabel("L values (when A=1)")
plt.ylabel("Frequency")
plt.title("Distribution of X for A=1")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
NC

In [ ]:
# Filter rows where A == 1
subset = DI[DI['A'] == 1]

# Loop through each variable in L
for var in picked_for_L:
    print(f"\n--- Variable: {var} ---")
    
    if pd.api.types.is_numeric_dtype(subset[var]):
        # For continuous variable: show min, max, mean
        print("Type: Continuous")
        print("Min:", subset[var].min())
        print("Max:", subset[var].max())
        print("Mean:", round(subset[var].mean(), 2))
    else:
        # For categorical variable: show value counts
        print("Type: Categorical")
        print(subset[var].value_counts(dropna=False))


# Discharge safe analysis

In [ ]:
DS_filtered = DS.loc[
    ~DS["admission_id"].isin(
        DS.loc[(DS["t0"] == 179) & (DS["A"] == 1), "admission_id"]
    )
].copy()


In [ ]:
(
    DS.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Py']
             .mean()
)

In [ ]:
result = (
    DS.loc[(DS["t0"] > 170) & (DS["A"] == 1)]
      .groupby("t0")["admission_id"]
      .nunique()
      .sort_values(ascending=False)
)

result


# Sanity checks

In [ ]:
mimicdata = mimicdata.rename(columns={'Y': 'Y_hat'})

In [86]:
NC = g.summary_dict['sim_data']['Natural course']
#D3 = g.summary_dict['sim_data']['Discharge 5th day']
#DS = g.summary_dict['sim_data']['Discharge Knight']

# Deaths

In [91]:
last_rows = (
        NC.sort_values(['admission_id', 't0'])
          .groupby('admission_id')
          .tail(1)
    )
    
I_hat_mean = last_rows['I_hat'].fillna(0).mean()
Z_hat_mean = last_rows['Z_hat'].fillna(0).mean()
Y_hat_mean = last_rows['Y_hat'].mean()

results = []
label = "NC"
results.append({
    "Strategy": label,
    "Mean I_hat (stay-level)": I_hat_mean,
    "Mean Z_hat (stay-level)": Z_hat_mean,
    "Mean Y_hat (stay-level)": Y_hat_mean
})
summary_table = pd.DataFrame(results)
summary_table

,Strategy,Mean I_hat (stay-level),Mean Z_hat (stay-level),Mean Y_hat (stay-level)
0,NC,0.78,0.016,0.796


# Comparison of covariate distributions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import ceil

# -----------------------------
# Helper: last row per admission
# -----------------------------
def last_rows(df, id_col="admission_id", time_col="t0"):
    return (df.sort_values([id_col, time_col])
              .groupby(id_col)
              .tail(1)
              .reset_index(drop=True))

# -----------------------------
# Ax-level plotting helpers
# -----------------------------
def ax_numeric_overlay(ax, m_vals, s_vals, name, bins=30):
    m_vals = pd.to_numeric(pd.Series(m_vals), errors="coerce").dropna().values
    s_vals = pd.to_numeric(pd.Series(s_vals), errors="coerce").dropna().values
    if len(m_vals) == 0 or len(s_vals) == 0:
        ax.set_visible(False)
        return None

    h1 = ax.hist(m_vals, bins=bins, density=True, alpha=0.5, label="mimic-iv")
    h2 = ax.hist(s_vals, bins=bins, density=True, alpha=0.5, label="natural course")
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    return ax.get_legend_handles_labels()

def ax_categorical_bars(ax, m_ser, s_ser, name, top_k=15):
    m_ser = pd.Series(m_ser).astype("object")
    s_ser = pd.Series(s_ser).astype("object")

    m_p = m_ser.value_counts(normalize=True, dropna=False)
    s_p = s_ser.value_counts(normalize=True, dropna=False)

    cats = list((m_p.add(s_p, fill_value=0))
                .sort_values(ascending=False)
                .head(top_k)
                .index)

    if len(cats) == 0:
        ax.set_visible(False)
        return None

    m_vals = [m_p.get(c, 0.0) for c in cats]
    s_vals = [s_p.get(c, 0.0) for c in cats]

    x = np.arange(len(cats))
    width = 0.4

    b1 = ax.bar(x - width/2, m_vals, width, label="mimic-iv")
    b2 = ax.bar(x + width/2, s_vals, width, label="natural course")

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in cats], rotation=45, ha="right", fontsize=7)
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis="y", labelsize=8)
    return ax.get_legend_handles_labels()

# -----------------------------
# Prepare data
# -----------------------------
m_last = last_rows(mimicdata)
s_last = last_rows(simdata)

# -----------------------------
# Define covariates
# -----------------------------
numeric_covariates = list(picked_L)

categorical_covariates = [
    "vent_mode__last__last_12h"
]

# -----------------------------
# Subgroup: in-ICU death only
# (Assumes D=1 indicates in-ICU death; keep A=0 to exclude discharged)
# -----------------------------
mask_m = (m_last["A"] == 1) & (m_last["Z"] == 1)
mask_s = (s_last["A"] == 1) & (s_last["Z_hat"] == 1)

m_sub = m_last.loc[mask_m]
s_sub = s_last.loc[mask_s]

print(f"IN-ICU death subgroup | mimic n={len(m_sub)} | sim n={len(s_sub)}")

# -----------------------------
# Build plot list in left-to-right order
# -----------------------------
plot_specs = []
for var in numeric_covariates:
    if var in m_sub.columns and var in s_sub.columns:
        plot_specs.append(("num", var))

for var in categorical_covariates:
    if var in m_sub.columns and var in s_sub.columns:
        plot_specs.append(("cat", var))

k = len(plot_specs)
if k == 0:
    raise ValueError("No covariates found in both datasets for the selected subgroup.")

# -----------------------------
# Grid layout (left-to-right)
# -----------------------------
ncols = 3
nrows = ceil(k / ncols)

fig_w = 5.2 * ncols
fig_h = 3.6 * nrows
fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=200, constrained_layout=True)
axes = np.array(axes).reshape(-1)

legend_handles = None
legend_labels = None

for i, (kind, var) in enumerate(plot_specs):
    ax = axes[i]
    if kind == "num":
        out = ax_numeric_overlay(
            ax,
            m_sub[var],
            s_sub[var],
            name=f"{var}"
        )
    else:
        out = ax_categorical_bars(
            ax,
            m_sub[var],
            s_sub[var],
            name=f"{var}"
        )

    # Capture legend handles once for a global legend
    if out is not None and legend_handles is None:
        legend_handles, legend_labels = out

# Hide unused axes
for j in range(k, len(axes)):
    axes[j].set_visible(False)

# Global title + global legend
fig.suptitle("Covariate distribution comparison at in-ICU death (last row per admission)", fontsize=12)

if legend_handles is not None:
    fig.legend(
        legend_handles, legend_labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.02)
    )

plt.show()


# SMD plots